# Orientation: Explore Your Routing Infrastructure

This notebook helps you explore what's already been set up in your Snowflake account for the **Fleet Intelligence with Cortex Code** quickstart.

Run each cell to inspect the pre-provisioned infrastructure — image repositories, container images, compute pools, warehouses, and services.

In [ ]:
%%sql -r session_context
SELECT CURRENT_ROLE() AS role,
       CURRENT_WAREHOUSE() AS warehouse,
       CURRENT_USER() AS username,
       CURRENT_ACCOUNT_NAME() AS account

## Database & Schemas
The **OPENROUTESERVICE_APP** database holds all routing infrastructure.

In [ ]:
%%sql -r ors_schemas
SHOW SCHEMAS IN DATABASE OPENROUTESERVICE_APP

## Image Repository
Container images are stored in the **CORE.IMAGE_REPOSITORY**.

| Image | Purpose |
|---|---|
| `openrouteservice` | The core ORS routing engine |
| `ors_control_app` | React + Express dashboard |
| `routing_reverse_proxy` | Nginx proxy |
| `vroom-docker` | VROOM solver |
| `downloader` | Downloads OSM map files |

In [ ]:
%%sql -r image_repos
SHOW IMAGE REPOSITORIES IN DATABASE OPENROUTESERVICE_APP

In [ ]:
%%sql -r images
SHOW IMAGES IN IMAGE REPOSITORY OPENROUTESERVICE_APP.CORE.IMAGE_REPOSITORY

## Compute Pools
Compute pools provide the VM nodes that run containerised services (SPCS).

In [ ]:
%%sql -r compute_pools
SHOW COMPUTE POOLS

## Available Instance Families

In [ ]:
%%sql -r instance_families
SHOW COMPUTE POOL INSTANCE FAMILIES

## Warehouses

In [ ]:
%%sql -r warehouses
SHOW WAREHOUSES

## Services (SPCS)
No services are deployed yet. To deploy, ask Cortex Code: **Build the routing solution**

In [ ]:
%%sql -r services
SHOW SERVICES IN DATABASE OPENROUTESERVICE_APP

## Stages

In [ ]:
%%sql -r stages
SHOW STAGES IN DATABASE OPENROUTESERVICE_APP

In [ ]:
%%sql -r table_row_counts
SELECT 'FACT_TRIPS' AS table_name, COUNT(*) AS row_count FROM SYNTHETIC_DATASETS.UNIFIED.FACT_TRIPS
UNION ALL
SELECT 'FACT_VEHICLE_TELEMETRY', COUNT(*) FROM SYNTHETIC_DATASETS.UNIFIED.FACT_VEHICLE_TELEMETRY
UNION ALL
SELECT 'DIM_FLEET', COUNT(*) FROM SYNTHETIC_DATASETS.UNIFIED.DIM_FLEET
UNION ALL
SELECT 'DIM_POIS', COUNT(*) FROM SYNTHETIC_DATASETS.UNIFIED.DIM_POIS

In [ ]:
%%sql -r trip_stats
SELECT
  AVG(DURATION_MINUTES) AS avg_duration_min,
  MIN(DURATION_MINUTES) AS min_duration_min,
  MAX(DURATION_MINUTES) AS max_duration_min,
  STDDEV(DURATION_MINUTES) AS stddev_duration_min,
  AVG(DISTANCE_KM) AS avg_distance_km,
  MIN(DISTANCE_KM) AS min_distance_km,
  MAX(DISTANCE_KM) AS max_distance_km,
  STDDEV(DISTANCE_KM) AS stddev_distance_km
FROM SYNTHETIC_DATASETS.UNIFIED.FACT_TRIPS

In [ ]:
%%sql -r busiest_vehicles
SELECT f.VEHICLE_ID, d.VEHICLE_TYPE, COUNT(*) AS trip_count
FROM SYNTHETIC_DATASETS.UNIFIED.FACT_TRIPS f
JOIN SYNTHETIC_DATASETS.UNIFIED.DIM_FLEET d ON f.VEHICLE_ID = d.VEHICLE_ID
GROUP BY f.VEHICLE_ID, d.VEHICLE_TYPE
ORDER BY trip_count DESC
LIMIT 10

In [ ]:
%%sql -r trips_by_dow
SELECT DAYNAME(TO_TIMESTAMP(TRIP_START)) AS day_of_week,
       DAYOFWEEK(TO_TIMESTAMP(TRIP_START)) AS day_num,
       COUNT(*) AS trip_count
FROM SYNTHETIC_DATASETS.UNIFIED.FACT_TRIPS
GROUP BY day_of_week, day_num
ORDER BY day_num

In [ ]:
%%sql -r poi_categories
SELECT CATEGORY, COUNT(*) AS poi_count
FROM SYNTHETIC_DATASETS.UNIFIED.DIM_POIS
GROUP BY CATEGORY
ORDER BY poi_count DESC
LIMIT 10

In [ ]:
%%sql -r trips_per_hour
SELECT HOUR(TO_TIMESTAMP(TRIP_START)) AS hour_of_day,
       COUNT(*) AS trip_count
FROM SYNTHETIC_DATASETS.UNIFIED.FACT_TRIPS
GROUP BY hour_of_day
ORDER BY hour_of_day

In [ ]:
%%sql -r cumulative_distance
SELECT DATE_TRUNC('hour', TO_TIMESTAMP(TRIP_START)) AS trip_hour,
       SUM(DISTANCE_KM) AS hourly_distance_km,
       SUM(SUM(DISTANCE_KM)) OVER (ORDER BY trip_hour) AS cumulative_distance_km
FROM SYNTHETIC_DATASETS.UNIFIED.FACT_TRIPS
GROUP BY trip_hour
ORDER BY trip_hour

In [ ]:
%%sql -r control_app_endpoint
SHOW ENDPOINTS IN SERVICE OPENROUTESERVICE_APP.CORE.ORS_CONTROL_APP;
SELECT 'https://' || "ingress_url" AS control_app_url FROM TABLE(RESULT_SCAN(LAST_QUERY_ID())) WHERE "name" = 'ors-control-app'

## Test Routing Functions

Now that services are running, let's test the core ORS routing functions. These are the building blocks for all fleet intelligence demos.

In [ ]:
%%sql -r health_check
SELECT
  CASE WHEN DISTANCE IS NOT NULL THEN 'HEALTHY' ELSE 'NOT READY' END AS ors_status,
  ROUND(DISTANCE, 0) AS test_distance_m,
  ROUND(DURATION, 0) AS test_duration_s
FROM TABLE(
  OPENROUTESERVICE_APP.CORE.DIRECTIONS(
    'driving-car',
    ARRAY_CONSTRUCT(-122.4194, 37.7749),
    ARRAY_CONSTRUCT(-122.4089, 37.7836)
  )
)

### Directions
Calculate a driving route between two San Francisco landmarks (Ferry Building → Golden Gate Park).

In [ ]:
%%sql -r directions_result
SELECT
  ROUND(DISTANCE, 0) AS distance_meters,
  ROUND(DURATION / 60, 1) AS duration_minutes,
  GEOJSON AS route_geometry
FROM TABLE(
  OPENROUTESERVICE_APP.CORE.DIRECTIONS(
    'driving-car',
    ARRAY_CONSTRUCT(-122.3937, 37.7955),
    ARRAY_CONSTRUCT(-122.4862, 37.7694)
  )
)

### Isochrones
Generate a 10-minute driving reachability polygon from Union Square.

In [ ]:
%%sql -r isochrone_result
SELECT
  RESPONSE:features[0]:properties:value::INT AS range_seconds,
  ROUND(RESPONSE:features[0]:properties:area::FLOAT / 1000000, 2) AS area_km2,
  GEOJSON AS isochrone_polygon
FROM TABLE(
  OPENROUTESERVICE_APP.CORE.ISOCHRONES(
    'driving-car',
    -122.4075::FLOAT,
    37.7881::FLOAT,
    10::INT
  )
)

### Route Optimization (VRP)
Optimize delivery of 4 packages from a depot using the VROOM solver.

In [ ]:
%%sql -r optimization_result
SELECT
  VEHICLE,
  ROUND(DURATION / 60, 1) AS route_duration_min,
  ARRAY_SIZE(STEPS) AS num_stops,
  GEOJSON AS route_geometry
FROM TABLE(
  OPENROUTESERVICE_APP.CORE.OPTIMIZATION(
    PARSE_JSON('[{"id":1,"location":[-122.4194,37.7749]},{"id":2,"location":[-122.4089,37.7836]},{"id":3,"location":[-122.4528,37.7694]},{"id":4,"location":[-122.3937,37.7955]}]')::ARRAY,
    PARSE_JSON('[{"id":1,"start":[-122.4312,37.7750],"end":[-122.4312,37.7750],"profile":"driving-car"}]')::ARRAY,
    ARRAY_CONSTRUCT()
  )
)

### Travel Time Matrix
Compute pairwise travel times between 3 locations (SFO, Downtown, Mission District).

In [ ]:
%%sql -r matrix_result
SELECT
  result:durations[0][0]::INT AS sfo_to_sfo_sec,
  result:durations[0][1]::INT AS sfo_to_downtown_sec,
  result:durations[0][2]::INT AS sfo_to_mission_sec,
  result:durations[1][0]::INT AS downtown_to_sfo_sec,
  result:durations[1][1]::INT AS downtown_to_downtown_sec,
  result:durations[1][2]::INT AS downtown_to_mission_sec,
  result:durations[2][0]::INT AS mission_to_sfo_sec,
  result:durations[2][1]::INT AS mission_to_downtown_sec,
  result:durations[2][2]::INT AS mission_to_mission_sec
FROM (
  SELECT OPENROUTESERVICE_APP.CORE.MATRIX(
    'driving-car',
    ARRAY_CONSTRUCT(
      ARRAY_CONSTRUCT(-122.3750, 37.6213),
      ARRAY_CONSTRUCT(-122.4194, 37.7749),
      ARRAY_CONSTRUCT(-122.4194, 37.7599)
    )
  ) AS result
)

## ORS Control App

We've now provisioned a full React-based Control App running on Snowpark Container Services. This dashboard lets you:

- **Manage regions** — add new cities, download OSM map files, build routing graphs
- **Test routing functions** — directions, isochrones, matrix, and optimization interactively
- **Generate fleet data** — use Data Studio to create realistic vehicle telemetry for any region
- **Run the Agent Playground** — guided demo scenarios powered by Snowflake Cortex Agent
- **Monitor services** — view service health, suspend/resume ORS instances

The cell below retrieves the public URL for the app we just deployed:

In [ ]:
%%sql -r app_url
SHOW ENDPOINTS IN SERVICE OPENROUTESERVICE_APP.CORE.ORS_CONTROL_APP


### Deploy a simple Fleet Explorer App

This deploys a **Streamlit app** with interactive maps, routing directions, and isochrone visualisations. It checks ORS service health before enabling routing features.

> **Prompt:**
>
> ```
> Create and deploy a Streamlit app in this workspace called "fleet-map" on warehouse DEFAULT_WH.
> Add pydeck to environment.yml (snowflake conda channel). Use get_active_session() for the connection.
> The app should have 3 tabs:
>
> Tab 1 - POI Map: Plot all POIs from SYNTHETIC_DATASETS.UNIFIED.DIM_POIS (columns: NAME, CATEGORY, LNG, LAT)
>   on a pydeck ScatterplotLayer with dark map style, with a category filter multiselect.
>
> Tab 2 - Directions: Let users pick 2 POIs from DIM_POIS, then call
>   OPENROUTESERVICE_APP.CORE.DIRECTIONS('driving-car', ARRAY_CONSTRUCT(lng1,lat1), ARRAY_CONSTRUCT(lng2,lat2))
>   and show the route on a GeoJsonLayer with distance/duration metrics.
>
> Tab 3 - Isochrones: Let users pick a POI and a travel time slider (1-15 min), then call
>   OPENROUTESERVICE_APP.CORE.ISOCHRONES('driving-car', lng::FLOAT, lat::FLOAT, minutes::NUMBER)
>   and show the reachability polygon on a GeoJsonLayer.
>
> Add a sidebar health check that calls OPENROUTESERVICE_APP.CORE.CHECK_HEALTH()
> Deploy the app to SYNTHETIC_DATASETS.UNIFIED.FLEET_MAP.
> ```

---

## ORS Control App (SPCS React Dashboard)

Now that you've experienced the core ORS routing functions (directions, isochrones, matrix, optimisation) through SQL and Streamlit, let's see the full potential.

The **ORS Control App** is a production-grade React + Express dashboard running on Snowpark Container Services. It provides:

| Feature | Description |
|---|---|
| **Region Management** | Add, remove, and monitor ORS map regions (cities) |
| **Service Health** | Real-time status of all SPCS services (ORS engine, gateway, VROOM, downloader) |
| **Interactive Routing** | Point-and-click directions, isochrones, and matrix on a full deck.gl map |
| **Fleet Monitoring** | Live vehicle positions, route replay, and telemetry overlays |
| **Configuration** | Change routing profiles, adjust engine settings, manage compute pools |

This app demonstrates what's possible when you combine ORS + SPCS + a modern frontend framework.


## Optional Add-On Skills

The core lab deploys routing infrastructure, the Cortex Agent, and the Agent Playground. The following **optional add-ons** extend the demo with additional analytics and visualisation. Each is a standalone skill — invoke from Cortex Code in this Workspace.

| Skill | What it adds | Demo scenario unlocked |
|-------|-------------|----------------------|
| `$add-weather-routing` | Met Office real-time weather — wind, rain, visibility before deploying vehicles | **Weather-Aware Routing** |
| `$add-pharma-intelligence` | Downstream pharmacy analytics — inventory, wastage, demand forecasting, replenishment | **Pharma Supply Intelligence** |
| `$add-pharma-supply-chain` | Upstream manufacturing — plants, suppliers, batches, shipments, material inventory | **Pharma Manufacturing** |
| `$add-plant-map` | Plant Intelligence map — Overture Maps 3D building footprints colour-coded by supply chain alerts | **Plant Intelligence** (visual map) |
| `$add-fleet-analytics` | Fleet trip + telemetry analytics — hourly demand, speeding, battery, dwell/idle time | **Fleet Analytics** |

### Dependency order

```
$routing-agent  (deployed as part of $build-routing-solution)
├── $add-weather-routing
├── $setup-agent-playground
│       └── $add-pharma-intelligence
├── $add-pharma-supply-chain
│       └── $add-plant-map
└── $add-fleet-analytics
```

### How to invoke

Type the skill name in Cortex Code — it guides you through prerequisites and runs deployment SQL automatically:

```
$add-pharma-supply-chain
```

In [ ]:
%%sql -r addon_status
-- Check which optional add-ons have been applied to this instance
SELECT 'add-weather-routing'    AS addon,
       'Met Office weather tool' AS description,
       IFF(COUNT(*) > 0, 'deployed', 'not deployed') AS status
FROM INFORMATION_SCHEMA.PROCEDURES
WHERE PROCEDURE_SCHEMA = 'ROUTING_AGENT' AND PROCEDURE_NAME = 'TOOL_WEATHER'
UNION ALL
SELECT 'add-pharma-intelligence', 'Pharmacy inventory + demand',
       IFF(COUNT(*) > 0, 'deployed', 'not deployed')
FROM FLEET_INTELLIGENCE.INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'ROUTING_AGENT' AND TABLE_NAME = 'SF_INVENTORY'
UNION ALL
SELECT 'add-pharma-supply-chain', 'Manufacturing plants + batches',
       IFF(COUNT(*) > 0, 'deployed', 'not deployed')
FROM FLEET_INTELLIGENCE.INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'PHARMA_SUPPLY_CHAIN' AND TABLE_NAME = 'PLANTS'
UNION ALL
SELECT 'add-plant-map', 'Overture building footprints',
       IFF(COUNT(*) > 0, 'deployed', 'not deployed')
FROM FLEET_INTELLIGENCE.INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'PHARMA_SUPPLY_CHAIN' AND TABLE_NAME = 'PLANT_BUILDING_FOOTPRINTS'
UNION ALL
SELECT 'add-fleet-analytics', 'Fleet trips + telemetry semantic views',
       IFF(COUNT(*) > 0, 'deployed', 'not deployed')
FROM FLEET_INTELLIGENCE.INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA = 'PUBLIC' AND TABLE_NAME = 'FLEET_TRIPS_SV'
ORDER BY addon